In [1]:
import pandas as pd
df = pd.read_csv('fra_cleaned.csv', sep=';', encoding='ISO-8859-1')

missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data_summary = pd.DataFrame({
    'Liczba braków': missing_values,
    'Procent braków (%)': missing_percentage.round(2)
})

columns_with_nan = missing_data_summary[missing_data_summary['Liczba braków'] > 0].sort_values(by='Liczba braków', ascending=False)

display(columns_with_nan)

,Liczba braków,Procent braków (%)
Perfumer2,22727,94.45
Year,2037,8.47
mainaccord5,981,4.08
mainaccord4,388,1.61
mainaccord3,114,0.47
mainaccord2,13,0.05


In [2]:
df['Rating Value'] = df['Rating Value'].astype(str).str.replace(',', '.').astype(float)
df['Perfumer2'] = df['Perfumer2'].fillna('')
df['Perfumers'] = df.apply(
    lambda row: f"{row['Perfumer1']}, {row['Perfumer2']}" if row['Perfumer2'] != '' else row['Perfumer1'], 
    axis=1
)
df = df.drop(columns=['Perfumer1', 'Perfumer2'])
columns_to_lower = [
    'Top', 'Middle', 'Base', 
    'mainaccord1', 'mainaccord2', 'mainaccord3', 'mainaccord4', 'mainaccord5'
]

for column in columns_to_lower:
    df[column] = df[column].astype(str).str.lower().str.strip()
    df[column] = df[column].replace(['nan', 'unknown', 'none', ''], pd.NA)

display(df[['Perfume', 'Rating Value', 'Perfumers', 'Year']].head())

,Perfume,Rating Value,Perfumers,Year
0,accento-overdose-pride-edition,1.42,unknown,2022.0
1,classique-pride-2024,1.86,unknown,2024.0
2,classique-pride-2023,1.91,"natalie gracia-cetto, quentin bisch",2023.0
3,pride-edition-man,1.92,unknown,2019.0
4,le-male-pride-collector,1.93,francis kurkdjian,2020.0


In [3]:
missing_accords = df[df['mainaccord2'].isna() | df['mainaccord3'].isna()]
columns_to_show = ['Perfume', 'Brand', 'mainaccord1', 'mainaccord2', 'mainaccord3']
display(missing_accords[columns_to_show].tail(10))

missing_accords_df = df[df['mainaccord2'].isna() | df['mainaccord3'].isna()].copy()
columns_to_export = ['Perfume', 'Brand', 'mainaccord1', 'mainaccord2', 'mainaccord3']
missing_accords_df[columns_to_export].to_excel('missing_accords.xlsx', index=True)
print("Liczba wyeksportowanych wierszy do pliku 'missing_accords.xlsx':", len(missing_accords_df))

,Perfume,Brand,mainaccord1,mainaccord2,mainaccord3
21939,verveine-agrumes-2015,l-occitane-en-provence,citrus,aromatic,NaN
22376,cedro-di-diamante,perris-monte-carlo,citrus,warm spicy,NaN
22737,fleur-d-oranger-intense,fragonard,white floral,citrus,NaN
23077,khashab-al-oud-bois-de-oud,abdul-samad-al-qurashi,oud,NaN,NaN
23100,mercedes-benz-cologne,mercedes-benz,citrus,aromatic,NaN
23633,cristal-zest,massimo-dutti,citrus,aromatic,NaN
23671,jaguar-for-men-oud,jaguar,woody,vanilla,NaN
23786,ohood,amouage,rose,powdery,NaN
23831,dahn-oud-ateeque,abdul-samad-al-qurashi,oud,NaN,NaN
23924,oud-precieux,reine-de-saba,oud,NaN,NaN


Liczba wyeksportowanych wierszy do pliku 'missing_accords.xlsx': 114


In [ ]:
import numpy as np

updated_accords_df = pd.read_excel('missing_accords_complete.xlsx', index_col=0)

cols_to_clean = ['mainaccord1', 'mainaccord2', 'mainaccord3']
for col in cols_to_clean:
    updated_accords_df[col] = updated_accords_df[col].astype(str).str.lower().str.strip()
    updated_accords_df[col] = updated_accords_df[col].replace(['nan', 'none', ''], np.nan)

for index, row in updated_accords_df.iterrows():
    for col in cols_to_clean:
        updated_accords_df.at[index, col] = row[col]

missing_accords2 = updated_accords_df['mainaccord2'].isna().sum()
missing_accords3 = updated_accords_df['mainaccord3'].isna().sum()

print(f"Brakujące wartości w mainaccord2 po aktualizacji: {missing_accords2}")
print(f"Brakujące wartości w mainaccord3 po aktualizacji: {missing_accords3}")

Brakujące wartości w mainaccord2 po aktualizacji: 0
Brakujące wartości w mainaccord3 po aktualizacji: 8
